# A Simple Diffusion Model on the MNIST Data Set using JAX / FLAX

This notebook is intended to store a simple implementation of a [diffusion model](https://en.wikipedia.org/wiki/Diffusion_model) as well as its training process on the [MNIST data set](https://www.tensorflow.org/datasets/catalog/mnist) using the Python libraries [JAX](https://docs.jax.dev/en/latest/index.html) and [FLAX](https://flax.readthedocs.io/en/stable/).

<center>
    <img src=imgs/jax.webp>
</center>

Let us first import the necessary data set. In this project, we are going to use the $(28 \times 28)$ MNIST images that are contained in the `tensorflow` package.

First of all, we are going to import the necessary modules.

In [1]:
# Basic libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import os

# For implementing the diffusion model
import jax
import jax.numpy as jnp
import flax.nnx as nnx

os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.3"  # do not occupy all GPU memory

2026-04-07 14:30:30.159641: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 14:30:40.687739: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Loading the MNIST Data

This section deals with loading the MNIST data set from the `tensorflow_datasets` package.

In [2]:
# Load the actual data
with tf.device("/cpu:0"):
    train_ds = tfds.load(name="mnist", split="train")
    X = np.array([batch["image"] for batch in train_ds.as_numpy_iterator()]).astype(np.float16)
    y = np.array([batch["label"] for batch in train_ds.as_numpy_iterator()]).astype(np.int8)

    # Normalize to interval [-1, 1]
    X = -1 + X * 2 / 255.0
    
    # Resize from (28 x 28) to (32 x 32)
    b, _, _, c = X.shape
    X = tf.image.resize(X, size=(32, 32), method="nearest").numpy()

X.shape, y.shape, X.max(), X.min()

I0000 00:00:1775565096.274971 3352652 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14188 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:05:00.0, compute capability: 8.9
2026-04-07 14:31:38.168481: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-04-07 14:31:44.886967: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-07 14:31:49.098282: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-07 14:31:49.576088: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


((60000, 32, 32, 1), (60000,), np.float16(1.0), np.float16(-1.0))

We have resized the images from $(28 \times 28)$ pixels to $(32 \times 32)$ pixels in order to make the downsampling steps more convenient later.

Let us display some of the images.

In [3]:
import ipywidgets

def plot(index):
    image = X[index]
    plt.imshow(image, cmap="gray")
    plt.title(f"Image of class {y[index]}")

ipywidgets.interact(plot, index=ipywidgets.IntSlider(min=0, max=X.shape[0], step=1))

interactive(children=(IntSlider(value=0, description='index', max=60000), Output()), _dom_classes=('widget-int…

<function __main__.plot(index)>

## Implementing the Diffusion Model

Let us first define some global constants.

In [4]:
# Random
rngs = nnx.Rngs(0)  # for deterministic results (training loop)

# Globals
TIME_EMB_DIM = 128  # global embedding dim
IMAGE_SHAPE = X[0].shape  # shape of base images

# Devices
cuda = jax.devices()[0]
cpu = jax.devices("cpu")[0]

# Training
TRAIN = True  # no training?
NUM_EPOCHS = 50
BATCH_SIZE = 64  # batch size (number of images per forward pass)
T_MAX = 500  # maximum timestep for diffusion model

### Implementing a Positional Encoding

In the following, we are going to implement a function that does the positional encoding used for the timesteps. This function realizes the [positional encoding proposed in the "Attention Is All You Need" paper by Vaswani et. al.](https://arxiv.org/pdf/1706.03762). It is defined as follows:

$$PE_{(pos, 2i)} = \sin(\frac{pos}{10000^{2i/d}})$$
$$PE_{(pos, 2i+1)} = \cos(\frac{pos}{10000^{2i/d}})$$

Here, $pos$ is the position to be encoded, i.e. the timestep, $i$ is the index and $d$ is the embedding or encoding dimension.

Now, let us implement the positional encoding.

In [5]:
class PositionalEncoding(nnx.Module):

    """
    A simple layer that performs positional encoding for one particular time step.
    """
    
    def __init__(self, embedding_dim=TIME_EMB_DIM):
        self.embedding_dim = embedding_dim

    @nnx.jit  # for compilation
    def __call__(self, pos, rngs=None):

        """
        Performs a positional encoding for one time step (pos).
        Params:
            - pos: A time step to be encoded (pos is an integer).
        Returns:
            Returns the positional encoding as a vector of size (embedding_dim,).
        """
        
        inner = pos / (
            10_000 ** (jnp.arange(start=0, stop=self.embedding_dim, step=2) / self.embedding_dim) + 1e-8
        )
        encoding = jnp.ones(shape=self.embedding_dim)

        # The following statements are a bit overcomplicated, but JAX arrays are immutable
        encoding = encoding.at[jnp.arange(start=0, stop=self.embedding_dim, step=2)].set(jnp.sin(inner))
        encoding = encoding.at[jnp.arange(start=1, stop=self.embedding_dim, step=2)].set(jnp.cos(inner))

        return encoding

class PositionalEncodingBatch(nnx.Module):

    """
    A simple layer that performs positional encoding for a batch of time steps.
    """
    
    def __init__(self, embedding_dim=TIME_EMB_DIM):
        self.pe = PositionalEncoding(embedding_dim=embedding_dim)

    @nnx.jit  # for compilation
    def __call__(self, pos, rngs=None):

        """
        The actual implementation of the layer.
        Params:
            - pos: A vector of position of size (batch_size,).
        Returns:
            This function returns the positional encodings as a matrix of size (batch_size, embedding_dim).
        """
        
        encoding = jnp.zeros(shape=(pos.size, self.pe.embedding_dim))
        for row in range(pos.size):
            encoding = encoding.at[row].set(self.pe(pos=pos[row]))
        return encoding

Let us test our implementation. We start with the simple `PositionalEncoding` for one number.

In [6]:
pe = PositionalEncoding()
pe(1).shape

(128,)

Now, let us test the batch version `PositionalEncodingBatch`.

In [7]:
pe_batch = PositionalEncodingBatch()
pe_batch(jnp.array([1, 2, 3])).shape

(3, 128)

So our positional encoding works fine. I have to admit, it is not the best way to implement it, but it should be quite intuitive.

### Implementing the Necessary Blocks

Let us now start implementing the diffusion model itself. We will stick to an unconditioned diffusion model only.


First, we will implement a simple residual block which processes and downsamples an image and gives it to the lower block. We will also implement an attention block which processes a given image by utilizing attention layers.

Let us start by implementing a simple abstract class for a diffusion model block.

In [8]:
class DiffusionModelBlock(nnx.Module):
    def __init__(self):
        self.lower = nnx.data(None)

    def set_lower(self, lower):
        self.lower = lower

    def __call__(self, X, timesteps, rngs=None):
        raise NotImplementedError("This is an abstract class.")

Let us now implement a residual block for our model which uses convolutional layers and group norms. It downsamples the image and gives it to the lower block.

In [9]:
class ResidualBlock(DiffusionModelBlock):

    """
    Implements a simple residual convolution block for the diffusion model.
    """
    
    def __init__(self,
                 image_shape,
                 in_features, out_features,
                 kernel_size=(3, 3),
                 strides=1,
                 padding=1,
                 use_bias=True,
                 activation=nnx.relu,
                 rngs=None,
                 out_activation=None
                ):

        # --- ATTRIBUTES ---
        
        self.image_shape = image_shape
        
        self.activation = nnx.data(activation)  # activation for the inner layers
        self.out_activation = self.activation if out_activation is None else out_activation  # possible activation for last layer
        self.out_features = out_features
        self.lower = nnx.data(None)

        self.peb = PositionalEncodingBatch()

        self.rngs = rngs
        

        # --- LAYERS ---

        self.time_mlp = nnx.Sequential(
            nnx.Linear(TIME_EMB_DIM, out_features, rngs=self.rngs),
            nnx.relu,
            nnx.Linear(out_features, out_features, rngs=self.rngs),
        )
        
        self.conv1 = nnx.Conv(
            in_features=in_features, out_features=out_features, kernel_size=kernel_size,
            strides=(1, 1), padding=padding, use_bias=use_bias, rngs=self.rngs,
            kernel_init=nnx.initializers.glorot_normal()
        )

        self.gn1 = nnx.GroupNorm(
            num_features=out_features, num_groups=None, group_size=8, rngs=self.rngs
        )

        self.conv2 = nnx.Conv(
            in_features=out_features, out_features=out_features, kernel_size=kernel_size,
            strides=(1, 1), padding=0, use_bias=use_bias, rngs=self.rngs,
            kernel_init=nnx.initializers.glorot_normal()
        )

        self.gn2 = nnx.GroupNorm(
            num_features=out_features, num_groups=None, group_size=8, rngs=self.rngs
        )

        self.upconv = nnx.ConvTranspose(
            in_features=out_features, out_features=out_features, kernel_size=(2, 2),
            strides=(1, 1), kernel_dilation=2, padding=2, use_bias=use_bias, rngs=self.rngs,
            kernel_init=nnx.initializers.glorot_normal()
        )

        self.gn3 = nnx.GroupNorm(
            num_features=out_features, num_groups=None, group_size=8, rngs=self.rngs
        )

        self.conv3 = nnx.Conv(
            in_features=out_features, out_features=out_features, kernel_size=kernel_size,
            strides=(1, 1), padding=padding, use_bias=use_bias, rngs=self.rngs,
            kernel_init=nnx.initializers.glorot_normal()
        )

        self.gn4 = nnx.GroupNorm(
            num_features=out_features, num_groups=None, group_size=8, rngs=self.rngs
        )

        self.conv4 = nnx.Conv(
            in_features=out_features, out_features=in_features, kernel_size=kernel_size,
            strides=(1, 1), padding=padding, use_bias=use_bias, rngs=self.rngs,
            kernel_init=nnx.initializers.glorot_normal()
        )

    @nnx.jit
    def __call__(self, X, timesteps, rngs=None):

        """
        Performs a forward pass of the ResidualBlock object.
        """
        
        # Timestep embedding
        temb = self.peb(timesteps)
        temb = self.time_mlp(temb)
        temb = temb[:, None, None, :]  # (B,1,1,C)

        out = self.activation(self.gn1(self.conv1(X)))
        out = out + temb  # adding time embedding

        skip = out  # remember for skip connection
        
        out = self.activation(self.gn2(self.conv2(out)))

        if self.lower is not None:
            out = self.lower(out, timesteps)
        
        out = self.activation(self.gn3(self.upconv(out)))

        out = self.activation(self.gn4(self.conv3(out + skip)))
        out = self.out_activation(self.conv4(out))
        
        return out

Let us quickly test the residual block.

In [10]:
rb = ResidualBlock(
    image_shape=IMAGE_SHAPE,
    in_features=1, out_features=64,
    rngs=nnx.Rngs(0)
)

rb.set_lower(
    ResidualBlock(
        image_shape=(30, 30, 64),
        in_features=64, out_features=128,
        rngs=nnx.Rngs(0)
    )
)

rb(jnp.ones(shape=(BATCH_SIZE, 32, 32, 1)), timesteps=jnp.ones(BATCH_SIZE)).shape  # works

(64, 32, 32, 1)

So the residual block works well. Let us now implement the attention block.

In [11]:
class AttentionBlock(DiffusionModelBlock):

    """
    Implements a simple residual attention block for the diffusion model.
    """
    
    def __init__(self,
                 image_shape,
                 num_heads=4,
                 rngs=None):

        # --- ATTRIBUTES ---
        self.image_shape = image_shape
        
        self.height, self.width, self.channels = self.image_shape
        
        self.num_heads = num_heads
        self.in_features = self.height * self.width
        self.lower = nnx.data(None)
        
        self.rngs = rngs


        # --- LAYERS ---
        self.attention1 = nnx.MultiHeadAttention(num_heads=self.num_heads, in_features=self.in_features,
                                                 decode=False, rngs=self.rngs,
                                                 kernel_init=nnx.initializers.glorot_normal())
        self.attention2 = nnx.MultiHeadAttention(num_heads=self.num_heads, in_features=self.in_features,
                                                 decode=False, rngs=self.rngs,
                                                 kernel_init=nnx.initializers.glorot_normal())

    @nnx.jit
    def __call__(self, X, timesteps, rngs=None):

        """
        Performs a forward pass of the AttentionBlock object.
        """

        # Spatial flattening
        out = jnp.reshape(a=X, shape=(BATCH_SIZE, self.in_features, self.channels))
        out = jnp.swapaxes(out, 1, 2)
        
        out = nnx.relu(self.attention1(out))

        # Spatial unflattening
        out = jnp.swapaxes(out, 1, 2)
        out = jnp.reshape(a=out, shape=(BATCH_SIZE, self.height, self.width, self.channels))

        out_temp = None if self.lower is not None else out
        if self.lower is not None:
            out_temp = X + self.lower(out, timesteps=timesteps)  # skip connection

        # Spatial flattening
        out = jnp.reshape(a=out_temp, shape=(BATCH_SIZE, self.in_features, self.channels))
        out = jnp.swapaxes(out, 1, 2)
        
        out = nnx.relu(self.attention2(out))

        # Spatial unflattening
        out = jnp.swapaxes(out, 1, 2)
        out = jnp.reshape(a=out, shape=(BATCH_SIZE, self.height, self.width, self.channels))

        return out_temp + out  # skip connection

Let us test the attention block.

In [12]:
ab = AttentionBlock(image_shape=(20, 20, 16), rngs=nnx.Rngs(0))
ab.lower = ResidualBlock(
    image_shape=(20, 20, 16),
    in_features=16, out_features=32,
    rngs=nnx.Rngs(0)
)

ab(jnp.ones(shape=(BATCH_SIZE, 20, 20, 16)), timesteps=jnp.ones(BATCH_SIZE)).shape

(64, 20, 20, 16)

So this also works properly.

### Implementing the Diffusion Model Itself

Let us now implement a class for the diffusion model itself. We have successfully implemented the necessary blocks and are now ready for the actual diffusion model class.

We have to define the positions of the residual and attention blocks as well as the number of channels for the corresponding layers.

Remember, that attention blocks *do not* change the channels or the spatial size.

In [13]:
channels = [
    1,  # for the input image
    128,
    256,
    256,
    256,
    256,
    256
]

a = "attention"
r = "residual"

layers = [
    r,
    r,
    r,
    a,
    r,
    a
]

assert len(layers) == len(channels) - 1

In [14]:
class DiffusionModel(nnx.Module):

    """
    Implements a simple diffusion model.
    """
    
    def __init__(self, channels, layers, rngs):

        # --- ATTRIBUTES ---
        self.image_shape = IMAGE_SHAPE
        h, _, _ = self.image_shape  # quadratic images (we only need the height)

        self.rngs = rngs


        # --- LAYERS ---
        if layers[0] != "residual":
            raise TypeError("Root should be a residual block!")
        self.root = ResidualBlock(
            image_shape=(h, h, channels[0]),
            in_features=channels[0], out_features=channels[1],
            out_activation=nnx.identity,
            rngs=self.rngs
        )

        # Reduce spatial size if layer was a convolutional block
        if layers[0] == "residual":
            h -= 2

        current = self.root

        for index in range(1, len(layers)):
            current.set_lower(
                ResidualBlock(
                    image_shape=(h, h, channels[index]),
                    in_features=channels[index], out_features=channels[index+1],
                    rngs=self.rngs
                ) if layers[index] == "residual" else AttentionBlock(
                    image_shape=(h, h, channels[index]),
                    rngs=self.rngs
                )
            )

            # Reduce spatial size if layer was a convolutional block
            if layers[index] == "residual":
                h -= 2

            # Go deeper through recursion
            current = current.lower

    @nnx.jit
    def __call__(self, X, timesteps, rngs=None):

        # Pass through recursive blocks
        return self.root(X, timesteps=timesteps, rngs=rngs)

Let us now instanciate our model.

In [15]:
dm = DiffusionModel(
    channels,
    layers,
    nnx.Rngs(0)
)

Let us pass a tiny amount of data through the model to see if it works properly.

In [16]:
dm(jnp.ones(shape=(BATCH_SIZE, 32, 32, 1)), timesteps=jnp.ones(BATCH_SIZE)).shape

(64, 32, 32, 1)

How many parameters does our model have? Let us see...

In [17]:
# Count parameters
params = nnx.state(dm, nnx.Param)
total_params = sum(np.prod(x.shape) for x in jax.tree.leaves(params))
total_params

np.int64(14298145)

Alright. Let us now implement the training loop for our model.

## Training our Diffusion Model

Let us now implement the training loop for our neural network.

First, let us implement the variance schedule.

In [18]:
variance_schedule = jnp.linspace(start=1e-4, stop=0.02, num=T_MAX)
alpha = 1 - variance_schedule
alpha_bar = jnp.cumprod(alpha)

Now, let us set up the training dataset as a `tensorflow` `Dataset`.

In [19]:
with tf.device("/cpu:0"):
    train_ds = tf.data.Dataset.from_tensor_slices((X,)).shuffle(1024).batch(BATCH_SIZE, drop_remainder=True)
train_ds

<_BatchDataset element_spec=(TensorSpec(shape=(64, 32, 32, 1), dtype=tf.float16, name=None),)>

Let us define the optimizer, the loss function and the necessary metrics. We are going to use the `optax` library for this.

In [20]:
import optax

learning_rate = 0.0005

opt = optax.chain(
    optax.clip_by_global_norm(20.0),
    optax.adamw(
        learning_rate=learning_rate
    )
)

optimizer = nnx.Optimizer(
    dm,
    opt,
    wrt=nnx.Param
)

metrics = nnx.MultiMetric(
  loss=nnx.metrics.Average('loss')
)

In [21]:
@nnx.jit
def loss_fn(model, rngs, X, timesteps, gaussians):

    """
    Computes the loss of the diffusion model.
    """
    
    preds = model(X, timesteps, rngs)
    mse = (preds - gaussians) ** 2
    mse = mse.mean(axis=(1,2,3))
    loss = mse.mean()
    return loss, preds

@nnx.jit
def train_step(model, optimizer, metrics, rngs, X, timesteps, gaussians):

    """
    Performs a single train step.
    """
    
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, preds), grads = grad_fn(model, rngs, X, timesteps, gaussians)
    metrics.update(loss=loss)
    optimizer.update(model, grads)

Now, let us implement the training loop for the diffusion model.

In [22]:
import datetime

if TRAIN:

    dm.train()

    metrics_history = {
      'train_loss': []
    }
    
    for epoch in range(NUM_EPOCHS):
        
        metrics.reset()
        
        for step, batch in enumerate(train_ds.as_numpy_iterator()):
    
            X = jnp.asarray(batch[0]).to_device(cuda)
    
            timesteps = rngs.randint(minval=0, maxval=T_MAX, shape=BATCH_SIZE)
            gaussians = rngs.normal((BATCH_SIZE, *IMAGE_SHAPE))
            alpha_bar_t = alpha_bar[timesteps]
            alpha_bar_t = alpha_bar_t.reshape(BATCH_SIZE, 1, 1, 1)
            
            X = jnp.sqrt(alpha_bar_t) * X + jnp.sqrt(1.0 - alpha_bar_t) * gaussians
            
            train_step(dm, optimizer, metrics, rngs, X, timesteps, gaussians)
    
            X = X.to_device(cpu)
            
        for metric, value in metrics.compute().items():
          metrics_history[f'train_{metric}'].append(value)
            
        if (epoch + 1) % 2 == 0:
            print(f"""Epoch: {epoch+1} | Loss: {metrics_history['train_loss'][epoch]:.4f}""")
            with open("dm.txt", mode="a") as file:
                file.write(f"Epoch {epoch+1} / {NUM_EPOCHS}, Loss {metrics_history['train_loss'][-1]}, Last update {datetime.datetime.now()}\n")
                file.close()

Finally, we are going to save our trained diffusion model.

In [26]:
import orbax.checkpoint as ocp
from time import sleep

checkpoint_dir_string = '/HOME1/users/students/simons/modules/diffusion_model/checkpoints/'

if TRAIN:

    checkpoint_dir = ocp.test_utils.erase_and_create_empty(
        checkpoint_dir_string
    )
    
    _, state = nnx.split(dm)
    pure_dict_state = nnx.to_pure_dict(state)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(checkpoint_dir / 'dm', state, force=True)
    
    sleep(10)

## Testing our Diffusion Model

Now it is time to test our trained diffusion model. So let us load it.

In [27]:
# Restore the checkpoint back to its `nnx.State` structure - need an abstract reference
abstract_model = nnx.eval_shape(
    lambda: DiffusionModel(
        channels,
        layers,
        rngs=nnx.Rngs(0)
    )
)
graphdef, abstract_state = nnx.split(abstract_model)
print('The abstract NNX state (all leaves are abstract arrays):')
nnx.display(abstract_state)

checkpointer = ocp.StandardCheckpointer()
state_restored = checkpointer.restore(checkpoint_dir_string + "/dm", abstract_state)
print('NNX State restored: ')
nnx.display(state_restored)

# The model is now good to use!
dm = nnx.merge(graphdef, state_restored)

The abstract NNX state (all leaves are abstract arrays):


FileNotFoundError: Checkpoint at /HOME1/users/students/simons/modules/diffusion_model/checkpoints/dm not found.